In [1]:
#!/usr/bin/env python3

"""
🌾 Crop Recommendation System with Area Code Support
=====================================================

Features:
✅ Crop Recommendation using Random Forest
✅ Top-5 Crop Ranking
✅ Rainfall Prediction using Linear Regression
✅ Soil/Environment Clustering using KMeans
✅ Area Code Categorical Encoding
✅ Model Evaluation
✅ Interactive Tkinter GUI
✅ Crop Probability Visualization
✅ Rainfall Stage Visualization
✅ Dataset Viewer

Author: Surya
"""

# ---------------------------------------------------------
# CPU Configuration
# ---------------------------------------------------------

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

# ---------------------------------------------------------
# Standard Libraries
# ---------------------------------------------------------

import tkinter as tk
from tkinter import ttk, messagebox
from tkinter import filedialog

# ---------------------------------------------------------
# Data Science Libraries
# ---------------------------------------------------------

import pandas as pd
import numpy as np

# ---------------------------------------------------------
# Machine Learning
# ---------------------------------------------------------

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    OrdinalEncoder
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans

from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    r2_score
)

from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# Visualization
# ---------------------------------------------------------

import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


In [5]:
# =========================================================
# DATASET LOADING
# =========================================================

DEFAULT_DATASET = "dataset.csv"


def load_dataset(file_path):
    """
    Load dataset from CSV file.
    """

    try:
        df = pd.read_csv(r"C:\Users\surya\Downloads\dataset.csv")

        print("=" * 60)
        print("📊 DATASET INFORMATION")
        print("=" * 60)

        print(f"Rows    : {df.shape[0]}")
        print(f"Columns : {df.shape[1]}")

        print("\n📌 Columns:")
        print(list(df.columns))

        print("\n📌 First 5 rows:")
        display(df.head())

        return df

    except FileNotFoundError:
        print(f"❌ Dataset not found: {file_path}")
        return None

    except Exception as e:
        print(f"❌ Error loading dataset: {e}")
        return None


df = load_dataset(DEFAULT_DATASET)

if df is None:
    raise FileNotFoundError(
        "dataset.csv was not found. "
        "Place dataset.csv in the same folder as model.ipynb."
    )

📊 DATASET INFORMATION
Rows    : 120
Columns : 9

📌 Columns:
['N', 'P', 'K', 'temperature', 'humidity', 'pH', 'rainfall', 'crop', 'area_code']

📌 First 5 rows:


,N,P,K,temperature,humidity,pH,rainfall,crop,area_code
0,81,54,51,29.2,74.7,5.7,191.8,coffee,500002
1,122,52,74,27.7,74.2,6.8,296.0,coffee,500001
2,44,24,23,19.6,65.2,7.0,198.5,rice,500001
3,101,61,74,24.3,73.0,6.7,165.8,maize,500001
4,90,58,30,22.5,72.0,7.4,226.7,maize,500001


In [6]:
# =========================================================
# DATASET VALIDATION
# =========================================================

required_columns = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "pH",
    "rainfall",
    "crop"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"❌ Missing required columns: {missing_columns}"
    )

print("✅ All required columns are available.")

if "area_code" in df.columns:
    print("📍 Area Code feature detected.")
else:
    print("⚠️ Area Code column not found.")
    print("The system will continue without Area Code.")

✅ All required columns are available.
📍 Area Code feature detected.


In [7]:
# =========================================================
# DATA CLEANING
# =========================================================

df = df.copy()

# Remove completely empty rows
df.dropna(how="all", inplace=True)

# Remove duplicate rows
duplicate_count = df.duplicated().sum()

if duplicate_count > 0:
    print(f"🧹 Removing {duplicate_count} duplicate rows...")
    df.drop_duplicates(inplace=True)

# Convert numerical columns to numeric
numeric_columns = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "pH",
    "rainfall"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# Remove rows where essential target values are missing
df.dropna(
    subset=[
        "N",
        "P",
        "K",
        "temperature",
        "humidity",
        "pH",
        "rainfall",
        "crop"
    ],
    inplace=True
)

# Area code handling
if "area_code" in df.columns:
    df["area_code"] = (
        df["area_code"]
        .astype(str)
        .str.strip()
    )

print("✅ Data cleaning completed.")
print(f"Final dataset shape: {df.shape}")

display(df.head())

✅ Data cleaning completed.
Final dataset shape: (120, 9)


,N,P,K,temperature,humidity,pH,rainfall,crop,area_code
0,81,54,51,29.2,74.7,5.7,191.8,coffee,500002
1,122,52,74,27.7,74.2,6.8,296.0,coffee,500001
2,44,24,23,19.6,65.2,7.0,198.5,rice,500001
3,101,61,74,24.3,73.0,6.7,165.8,maize,500001
4,90,58,30,22.5,72.0,7.4,226.7,maize,500001


In [8]:
# =========================================================
# FEATURE DEFINITIONS
# =========================================================

BASE_FEATURES = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "pH"
]

# Crop prediction uses rainfall
CROP_FEATURES = BASE_FEATURES + ["rainfall"]

# Rainfall prediction must NOT use rainfall
RAINFALL_FEATURES = BASE_FEATURES.copy()

# Add Area Code if available
if "area_code" in df.columns:
    CROP_FEATURES.append("area_code")
    RAINFALL_FEATURES.append("area_code")

print("🌱 Crop Features:")
print(CROP_FEATURES)

print("\n🌧️ Rainfall Features:")
print(RAINFALL_FEATURES)

🌱 Crop Features:
['N', 'P', 'K', 'temperature', 'humidity', 'pH', 'rainfall', 'area_code']

🌧️ Rainfall Features:
['N', 'P', 'K', 'temperature', 'humidity', 'pH', 'area_code']


In [9]:
# =========================================================
# AREA CODE ENCODING
# =========================================================

area_encoder = None

if "area_code" in df.columns:

    area_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    df["area_code_encoded"] = area_encoder.fit_transform(
        df[["area_code"]]
    ).astype(float)

    print("✅ Area Code encoding completed.")

    print("\n📍 Area Codes:")
    print(
        df[
            ["area_code", "area_code_encoded"]
        ]
        .drop_duplicates()
        .sort_values("area_code")
        .head(20)
    )

else:

    df["area_code_encoded"] = 0.0
    print("⚠️ No Area Code found. Using default value 0.")

✅ Area Code encoding completed.

📍 Area Codes:
  area_code  area_code_encoded
1    500001                0.0
0    500002                1.0
5    500003                2.0


In [10]:
# =========================================================
# PREPARE DATA FOR MACHINE LEARNING
# =========================================================

# ---------------------------------------------------------
# Crop Classification Data
# ---------------------------------------------------------

crop_model_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "pH",
    "rainfall",
    "area_code_encoded"
]

X_crop = df[crop_model_features].copy()

crop_encoder = LabelEncoder()

y_crop = crop_encoder.fit_transform(
    df["crop"].astype(str)
)

print(f"🌱 Number of crops: {len(crop_encoder.classes_)}")

print("\nCrops:")
for i, crop in enumerate(crop_encoder.classes_):
    print(f"{i}: {crop}")


# ---------------------------------------------------------
# Rainfall Regression Data
# ---------------------------------------------------------

rainfall_model_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "pH",
    "area_code_encoded"
]

X_rainfall = df[rainfall_model_features].copy()

y_rainfall = df["rainfall"].copy()


print("\n🌧️ Rainfall model features:")
print(rainfall_model_features)

🌱 Number of crops: 4

Crops:
0: coffee
1: maize
2: rice
3: sugarcane

🌧️ Rainfall model features:
['N', 'P', 'K', 'temperature', 'humidity', 'pH', 'area_code_encoded']


In [11]:
# =========================================================
# RANDOM FOREST CROP CLASSIFIER
# =========================================================

X_train_crop, X_test_crop, y_train_crop, y_test_crop = train_test_split(
    X_crop,
    y_crop,
    test_size=0.20,
    random_state=42,
    stratify=y_crop
)

crop_scaler = StandardScaler()

X_train_crop_scaled = crop_scaler.fit_transform(
    X_train_crop
)

X_test_crop_scaled = crop_scaler.transform(
    X_test_crop
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=1
)

rf_model.fit(
    X_train_crop_scaled,
    y_train_crop
)

# Evaluation
crop_predictions = rf_model.predict(
    X_test_crop_scaled
)

crop_accuracy = accuracy_score(
    y_test_crop,
    crop_predictions
)

print("=" * 60)
print("🌱 RANDOM FOREST RESULTS")
print("=" * 60)

print(
    f"Classification Accuracy: "
    f"{crop_accuracy * 100:.2f}%"
)

🌱 RANDOM FOREST RESULTS
Classification Accuracy: 16.67%


In [12]:
# =========================================================
# RAINFALL PREDICTION - LINEAR REGRESSION
# =========================================================

X_train_rain, X_test_rain, y_train_rain, y_test_rain = train_test_split(
    X_rainfall,
    y_rainfall,
    test_size=0.20,
    random_state=42
)

rainfall_scaler = StandardScaler()

X_train_rain_scaled = rainfall_scaler.fit_transform(
    X_train_rain
)

X_test_rain_scaled = rainfall_scaler.transform(
    X_test_rain
)

rainfall_model = LinearRegression()

rainfall_model.fit(
    X_train_rain_scaled,
    y_train_rain
)

rainfall_predictions = rainfall_model.predict(
    X_test_rain_scaled
)

rainfall_mae = mean_absolute_error(
    y_test_rain,
    rainfall_predictions
)

rainfall_r2 = r2_score(
    y_test_rain,
    rainfall_predictions
)

print("=" * 60)
print("🌧️ RAINFALL REGRESSION RESULTS")
print("=" * 60)

print(f"MAE : {rainfall_mae:.2f} mm")
print(f"R²  : {rainfall_r2:.4f}")

🌧️ RAINFALL REGRESSION RESULTS
MAE : 50.23 mm
R²  : -0.0092


In [13]:
# =========================================================
# KMEANS CLUSTERING
# =========================================================

cluster_features = [
    "N",
    "P",
    "K",
    "temperature",
    "humidity",
    "pH",
    "rainfall",
    "area_code_encoded"
]

X_cluster = df[cluster_features].copy()

cluster_scaler = StandardScaler()

X_cluster_scaled = cluster_scaler.fit_transform(
    X_cluster
)

kmeans_model = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans_model.fit_predict(
    X_cluster_scaled
)

df["cluster"] = cluster_labels

print("=" * 60)
print("🔶 KMEANS CLUSTERING")
print("=" * 60)

print(
    df["cluster"]
    .value_counts()
    .sort_index()
)

print("\n✅ Clustering completed.")

🔶 KMEANS CLUSTERING
cluster
0    30
1    35
2    26
3    29
Name: count, dtype: int64

✅ Clustering completed.


In [14]:
# =========================================================
# CLUSTER ANALYSIS
# =========================================================

cluster_summary = (
    df.groupby("cluster")[
        [
            "N",
            "P",
            "K",
            "temperature",
            "humidity",
            "pH",
            "rainfall"
        ]
    ]
    .mean()
    .round(2)
)

print("🔶 Cluster Characteristics:")

display(cluster_summary)

🔶 Cluster Characteristics:


,N,P,K,temperature,humidity,pH,rainfall
cluster,,,,,,,
0,54.60,41.50,31.87,24.42,74.82,6.43,202.11
1,89.83,34.63,51.23,28.24,68.11,6.66,168.03
2,96.31,52.27,52.81,23.67,78.83,7.17,190.73
3,78.28,55.66,61.62,29.89,79.81,6.28,179.40


In [15]:
# =========================================================
# PREDICTION ENGINE
# =========================================================

def encode_area_code(area_code):
    """
    Convert Area Code into numerical representation.
    Unknown Area Codes receive -1.
    """

    if area_encoder is None:
        return 0.0

    area_code = str(area_code).strip()

    encoded = area_encoder.transform(
        [[area_code]]
    )[0][0]

    return float(encoded)


def predict_rainfall(
    N,
    P,
    K,
    temperature,
    humidity,
    pH,
    area_code
):
    """
    Predict rainfall without using actual rainfall
    as an input.
    """

    area_encoded = encode_area_code(
        area_code
    )

    values = np.array([[
        N,
        P,
        K,
        temperature,
        humidity,
        pH,
        area_encoded
    ]])

    values_scaled = rainfall_scaler.transform(
        values
    )

    prediction = rainfall_model.predict(
        values_scaled
    )[0]

    # Prevent unrealistic negative rainfall
    prediction = max(0, float(prediction))

    return round(prediction, 2)


def predict_crop(
    N,
    P,
    K,
    temperature,
    humidity,
    pH,
    rainfall,
    area_code
):
    """
    Predict and rank crops.
    """

    area_encoded = encode_area_code(
        area_code
    )

    values = np.array([[
        N,
        P,
        K,
        temperature,
        humidity,
        pH,
        rainfall,
        area_encoded
    ]])

    values_scaled = crop_scaler.transform(
        values
    )

    probabilities = rf_model.predict_proba(
        values_scaled
    )[0]

    crop_names = crop_encoder.classes_

    rankings = list(
        zip(
            crop_names,
            probabilities
        )
    )

    rankings.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top_5 = [
        (
            crop,
            round(probability * 100, 2)
        )
        for crop, probability
        in rankings[:5]
    ]

    return top_5


def predict_cluster(
    N,
    P,
    K,
    temperature,
    humidity,
    pH,
    rainfall,
    area_code
):
    """
    Predict the closest environmental cluster.
    """

    area_encoded = encode_area_code(
        area_code
    )

    values = np.array([[
        N,
        P,
        K,
        temperature,
        humidity,
        pH,
        rainfall,
        area_encoded
    ]])

    values_scaled = cluster_scaler.transform(
        values
    )

    cluster = int(
        kmeans_model.predict(
            values_scaled
        )[0]
    )

    return cluster

In [16]:
# =========================================================
# PREDICTION ENGINE
# =========================================================

def encode_area_code(area_code):
    """
    Convert Area Code into numerical representation.
    Unknown Area Codes receive -1.
    """

    if area_encoder is None:
        return 0.0

    area_code = str(area_code).strip()

    encoded = area_encoder.transform(
        [[area_code]]
    )[0][0]

    return float(encoded)


def predict_rainfall(
    N,
    P,
    K,
    temperature,
    humidity,
    pH,
    area_code
):
    """
    Predict rainfall without using actual rainfall
    as an input.
    """

    area_encoded = encode_area_code(
        area_code
    )

    values = np.array([[
        N,
        P,
        K,
        temperature,
        humidity,
        pH,
        area_encoded
    ]])

    values_scaled = rainfall_scaler.transform(
        values
    )

    prediction = rainfall_model.predict(
        values_scaled
    )[0]

    # Prevent unrealistic negative rainfall
    prediction = max(0, float(prediction))

    return round(prediction, 2)


def predict_crop(
    N,
    P,
    K,
    temperature,
    humidity,
    pH,
    rainfall,
    area_code
):
    """
    Predict and rank crops.
    """

    area_encoded = encode_area_code(
        area_code
    )

    values = np.array([[
        N,
        P,
        K,
        temperature,
        humidity,
        pH,
        rainfall,
        area_encoded
    ]])

    values_scaled = crop_scaler.transform(
        values
    )

    probabilities = rf_model.predict_proba(
        values_scaled
    )[0]

    crop_names = crop_encoder.classes_

    rankings = list(
        zip(
            crop_names,
            probabilities
        )
    )

    rankings.sort(
        key=lambda x: x[1],
        reverse=True
    )

    top_5 = [
        (
            crop,
            round(probability * 100, 2)
        )
        for crop, probability
        in rankings[:5]
    ]

    return top_5


def predict_cluster(
    N,
    P,
    K,
    temperature,
    humidity,
    pH,
    rainfall,
    area_code
):
    """
    Predict the closest environmental cluster.
    """

    area_encoded = encode_area_code(
        area_code
    )

    values = np.array([[
        N,
        P,
        K,
        temperature,
        humidity,
        pH,
        rainfall,
        area_encoded
    ]])

    values_scaled = cluster_scaler.transform(
        values
    )

    cluster = int(
        kmeans_model.predict(
            values_scaled
        )[0]
    )

    return cluster

In [17]:
# =========================================================
# RAINFALL STAGE DISTRIBUTION
# =========================================================

def calculate_stage_rainfall(total_rainfall):

    stages = {
        "🌱 Seedling": 0.20,
        "🌿 Vegetative": 0.30,
        "🌼 Flowering": 0.30,
        "🌾 Maturity": 0.20
    }

    result = {}

    for stage, percentage in stages.items():

        result[stage] = round(
            total_rainfall * percentage,
            2
        )

    return result

In [18]:
# =========================================================
# MODERN TKINTER GUI
# =========================================================

class CropRecommendationApp:

    def __init__(self, root):

        self.root = root

        self.root.title(
            "🌾 Smart Crop Recommendation System"
        )

        self.root.geometry(
            "1250x900"
        )

        self.root.minsize(
            1100,
            750
        )

        # -------------------------------------------------
        # Variables
        # -------------------------------------------------

        self.vars = {}

        self.create_styles()
        self.create_header()
        self.create_input_section()
        self.create_button_section()
        self.create_result_section()

    # =====================================================
    # STYLES
    # =====================================================

    def create_styles(self):

        style = ttk.Style()

        try:
            style.theme_use("clam")
        except:
            pass

        style.configure(
            "Title.TLabel",
            font=("Segoe UI", 24, "bold")
        )

        style.configure(
            "Subtitle.TLabel",
            font=("Segoe UI", 11)
        )

        style.configure(
            "Section.TLabelframe",
            padding=12
        )

        style.configure(
            "Section.TLabelframe.Label",
            font=("Segoe UI", 12, "bold")
        )

        style.configure(
            "Action.TButton",
            font=("Segoe UI", 11, "bold"),
            padding=10
        )

    # =====================================================
    # HEADER
    # =====================================================

    def create_header(self):

        header = ttk.Frame(
            self.root,
            padding=15
        )

        header.pack(
            fill=tk.X
        )

        ttk.Label(
            header,
            text="🌾 Smart Crop Recommendation System",
            style="Title.TLabel"
        ).pack()

        ttk.Label(
            header,
            text=(
                "AI & Machine Learning Based "
                "Agricultural Decision Support"
            ),
            style="Subtitle.TLabel"
        ).pack(
            pady=(4, 0)
        )

    # =====================================================
    # INPUT SECTION
    # =====================================================

    def create_input_section(self):

        frame = ttk.LabelFrame(
            self.root,
            text="🌱 Soil & Environmental Parameters",
            style="Section.TLabelframe"
        )

        frame.pack(
            fill=tk.X,
            padx=20,
            pady=10
        )

        # -------------------------------------------------
        # Area Code
        # -------------------------------------------------

        ttk.Label(
            frame,
            text="📍 Area Code:"
        ).grid(
            row=0,
            column=0,
            padx=10,
            pady=8,
            sticky="w"
        )

        self.area_entry = ttk.Entry(
            frame,
            width=25
        )

        self.area_entry.grid(
            row=0,
            column=1,
            padx=10,
            pady=8,
            sticky="w"
        )

        # Put first known area code as example
        if "area_code" in df.columns:

            first_area = str(
                df["area_code"].iloc[0]
            )

            self.area_entry.insert(
                0,
                first_area
            )

        # -------------------------------------------------
        # Parameters
        # -------------------------------------------------

        parameters = {
            "N": (0, 140, 70),
            "P": (0, 100, 40),
            "K": (0, 100, 40),
            "temperature": (10, 45, 25),
            "humidity": (20, 100, 70),
            "pH": (3, 10, 6.5)
        }

        row = 1

        for name, (
            minimum,
            maximum,
            default
        ) in parameters.items():

            ttk.Label(
                frame,
                text=name.capitalize() + ":"
            ).grid(
                row=row,
                column=0,
                padx=10,
                pady=8,
                sticky="w"
            )

            variable = tk.DoubleVar(
                value=default
            )

            slider = ttk.Scale(
                frame,
                from_=minimum,
                to=maximum,
                variable=variable,
                orient="horizontal",
                length=450
            )

            slider.grid(
                row=row,
                column=1,
                padx=10,
                pady=8
            )

            value_label = ttk.Label(
                frame,
                textvariable=variable,
                width=8
            )

            value_label.grid(
                row=row,
                column=2,
                padx=10
            )

            self.vars[name] = variable

            row += 1

    # =====================================================
    # BUTTONS
    # =====================================================

    def create_button_section(self):

        frame = ttk.Frame(
            self.root,
            padding=10
        )

        frame.pack()

        ttk.Button(
            frame,
            text="🚀 Predict All",
            style="Action.TButton",
            command=self.predict
        ).pack(
            side=tk.LEFT,
            padx=8
        )

        ttk.Button(
            frame,
            text="🔄 Reset",
            style="Action.TButton",
            command=self.reset
        ).pack(
            side=tk.LEFT,
            padx=8
        )

        ttk.Button(
            frame,
            text="📊 Show Dataset",
            style="Action.TButton",
            command=self.show_dataset
        ).pack(
            side=tk.LEFT,
            padx=8
        )

    # =====================================================
    # RESULTS
    # =====================================================

    def create_result_section(self):

        self.notebook = ttk.Notebook(
            self.root
        )

        self.notebook.pack(
            fill=tk.BOTH,
            expand=True,
            padx=20,
            pady=10
        )

        # -------------------------------------------------
        # Summary Tab
        # -------------------------------------------------

        self.summary_tab = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            self.summary_tab,
            text="🏆 Results"
        )

        self.result_text = tk.Text(
            self.summary_tab,
            font=("Segoe UI", 11),
            wrap=tk.WORD,
            padx=15,
            pady=15
        )

        self.result_text.pack(
            fill=tk.BOTH,
            expand=True
        )

        # -------------------------------------------------
        # Crop Chart Tab
        # -------------------------------------------------

        self.crop_chart_tab = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            self.crop_chart_tab,
            text="📈 Crop Ranking"
        )

        # -------------------------------------------------
        # Rainfall Chart Tab
        # -------------------------------------------------

        self.rain_chart_tab = ttk.Frame(
            self.notebook
        )

        self.notebook.add(
            self.rain_chart_tab,
            text="🌧️ Rainfall"
        )

    # =====================================================
    # PREDICTION
    # =====================================================

    def predict(self):

        try:

            area_code = (
                self.area_entry
                .get()
                .strip()
            )

            if not area_code:

                messagebox.showwarning(
                    "Input Required",
                    "Please enter an Area Code."
                )

                return

            # ---------------------------------------------
            # Get inputs
            # ---------------------------------------------

            N = self.vars["N"].get()
            P = self.vars["P"].get()
            K = self.vars["K"].get()

            temperature = (
                self.vars["temperature"].get()
            )

            humidity = (
                self.vars["humidity"].get()
            )

            pH = self.vars["pH"].get()

            # ---------------------------------------------
            # Rainfall prediction
            # ---------------------------------------------

            predicted_rainfall = predict_rainfall(
                N,
                P,
                K,
                temperature,
                humidity,
                pH,
                area_code
            )

            # ---------------------------------------------
            # Crop prediction
            # ---------------------------------------------

            crop_ranking = predict_crop(
                N,
                P,
                K,
                temperature,
                humidity,
                pH,
                predicted_rainfall,
                area_code
            )

            top_crop = crop_ranking[0][0]

            # ---------------------------------------------
            # Cluster
            # ---------------------------------------------

            cluster = predict_cluster(
                N,
                P,
                K,
                temperature,
                humidity,
                pH,
                predicted_rainfall,
                area_code
            )

            # ---------------------------------------------
            # Stage rainfall
            # ---------------------------------------------

            stage_rainfall = (
                calculate_stage_rainfall(
                    predicted_rainfall
                )
            )

            # ---------------------------------------------
            # Display results
            # ---------------------------------------------

            self.result_text.delete(
                "1.0",
                tk.END
            )

            self.result_text.insert(
                tk.END,
                "🌾 CROP RECOMMENDATION RESULTS\n"
            )

            self.result_text.insert(
                tk.END,
                "=" * 65 + "\n\n"
            )

            self.result_text.insert(
                tk.END,
                f"📍 Area Code       : {area_code}\n"
            )

            self.result_text.insert(
                tk.END,
                f"🏆 Recommended Crop: {top_crop}\n"
            )

            self.result_text.insert(
                tk.END,
                f"🌧️ Predicted Rainfall: "
                f"{predicted_rainfall} mm\n"
            )

            self.result_text.insert(
                tk.END,
                f"🔶 Environmental Cluster: "
                f"{cluster}\n\n"
            )

            # ---------------------------------------------
            # Crop ranking
            # ---------------------------------------------

            self.result_text.insert(
                tk.END,
                "🌱 TOP 5 CROP RECOMMENDATIONS\n"
            )

            self.result_text.insert(
                tk.END,
                "-" * 65 + "\n"
            )

            for index, (
                crop,
                probability
            ) in enumerate(
                crop_ranking,
                start=1
            ):

                self.result_text.insert(
                    tk.END,
                    f"{index}. "
                    f"{crop:<30}"
                    f"{probability:>6.2f}%\n"
                )

            # ---------------------------------------------
            # Rainfall stages
            # ---------------------------------------------

            self.result_text.insert(
                tk.END,
                "\n🌧️ RAINFALL DISTRIBUTION BY STAGE\n"
            )

            self.result_text.insert(
                tk.END,
                "-" * 65 + "\n"
            )

            for stage, value in (
                stage_rainfall.items()
            ):

                self.result_text.insert(
                    tk.END,
                    f"{stage:<25}"
                    f"{value:>8.2f} mm\n"
                )

            # ---------------------------------------------
            # Input summary
            # ---------------------------------------------

            self.result_text.insert(
                tk.END,
                "\n📊 INPUT SUMMARY\n"
            )

            self.result_text.insert(
                tk.END,
                "-" * 65 + "\n"
            )

            self.result_text.insert(
                tk.END,
                f"N          : {N:.2f}\n"
            )

            self.result_text.insert(
                tk.END,
                f"P          : {P:.2f}\n"
            )

            self.result_text.insert(
                tk.END,
                f"K          : {K:.2f}\n"
            )

            self.result_text.insert(
                tk.END,
                f"Temperature: {temperature:.2f} °C\n"
            )

            self.result_text.insert(
                tk.END,
                f"Humidity   : {humidity:.2f}%\n"
            )

            self.result_text.insert(
                tk.END,
                f"pH         : {pH:.2f}\n"
            )

            # ---------------------------------------------
            # Charts
            # ---------------------------------------------

            self.create_crop_chart(
                crop_ranking
            )

            self.create_rainfall_chart(
                stage_rainfall
            )

            # ---------------------------------------------
            # Show result tab
            # ---------------------------------------------

            self.notebook.select(
                self.summary_tab
            )

        except Exception as e:

            messagebox.showerror(
                "Prediction Error",
                f"Unable to generate prediction:\n\n{e}"
            )

    # =====================================================
    # CROP CHART
    # =====================================================

    def create_crop_chart(
        self,
        crop_ranking
    ):

        for widget in (
            self.crop_chart_tab.winfo_children()
        ):

            widget.destroy()

        crops = [
            item[0]
            for item in crop_ranking
        ]

        probabilities = [
            item[1]
            for item in crop_ranking
        ]

        figure = plt.Figure(
            figsize=(9, 5),
            dpi=100
        )

        ax = figure.add_subplot(111)

        ax.bar(
            crops,
            probabilities
        )

        ax.set_title(
            "Top 5 Crop Recommendations"
        )

        ax.set_xlabel(
            "Crop"
        )

        ax.set_ylabel(
            "Probability (%)"
        )

        ax.tick_params(
            axis="x",
            rotation=30
        )

        figure.tight_layout()

        canvas = FigureCanvasTkAgg(
            figure,
            master=self.crop_chart_tab
        )

        canvas.draw()

        canvas.get_tk_widget().pack(
            fill=tk.BOTH,
            expand=True
        )

    # =====================================================
    # RAINFALL CHART
    # =====================================================

    def create_rainfall_chart(
        self,
        stage_rainfall
    ):

        for widget in (
            self.rain_chart_tab.winfo_children()
        ):

            widget.destroy()

        stages = list(
            stage_rainfall.keys()
        )

        values = list(
            stage_rainfall.values()
        )

        figure = plt.Figure(
            figsize=(9, 5),
            dpi=100
        )

        ax = figure.add_subplot(111)

        ax.bar(
            stages,
            values
        )

        ax.set_title(
            "Predicted Rainfall Distribution"
        )

        ax.set_xlabel(
            "Growth Stage"
        )

        ax.set_ylabel(
            "Rainfall (mm)"
        )

        ax.tick_params(
            axis="x",
            rotation=20
        )

        figure.tight_layout()

        canvas = FigureCanvasTkAgg(
            figure,
            master=self.rain_chart_tab
        )

        canvas.draw()

        canvas.get_tk_widget().pack(
            fill=tk.BOTH,
            expand=True
        )

    # =====================================================
    # RESET
    # =====================================================

    def reset(self):

        defaults = {
            "N": 70,
            "P": 40,
            "K": 40,
            "temperature": 25,
            "humidity": 70,
            "pH": 6.5
        }

        for key, value in defaults.items():

            self.vars[key].set(
                value
            )

        self.result_text.delete(
            "1.0",
            tk.END
        )

        self.result_text.insert(
            tk.END,
            "🔄 Inputs have been reset.\n\n"
            "Enter your Area Code and "
            "click 'Predict All' to begin."
        )

    # =====================================================
    # DATASET VIEWER
    # =====================================================

    def show_dataset(self):

        window = tk.Toplevel(
            self.root
        )

        window.title(
            "📊 Dataset Viewer"
        )

        window.geometry(
            "1100x650"
        )

        frame = ttk.Frame(
            window,
            padding=10
        )

        frame.pack(
            fill=tk.BOTH,
            expand=True
        )

        tree = ttk.Treeview(
            frame,
            show="headings"
        )

        # Scrollbars
        vertical_scroll = ttk.Scrollbar(
            frame,
            orient="vertical",
            command=tree.yview
        )

        horizontal_scroll = ttk.Scrollbar(
            frame,
            orient="horizontal",
            command=tree.xview
        )

        tree.configure(
            yscrollcommand=vertical_scroll.set,
            xscrollcommand=horizontal_scroll.set
        )

        tree.pack(
            side=tk.LEFT,
            fill=tk.BOTH,
            expand=True
        )

        vertical_scroll.pack(
            side=tk.RIGHT,
            fill=tk.Y
        )

        horizontal_scroll.pack(
            side=tk.BOTTOM,
            fill=tk.X
        )

        # Columns
        columns = list(
            df.columns
        )

        tree["columns"] = columns

        for column in columns:

            tree.heading(
                column,
                text=column
            )

            tree.column(
                column,
                width=120,
                anchor="center"
            )

        # First 100 rows
        for _, row in df.head(100).iterrows():

            tree.insert(
                "",
                tk.END,
                values=[
                    row[column]
                    for column in columns
                ]
            )


# =========================================================
# START APPLICATION
# =========================================================

def launch_application():

    root = tk.Tk()

    app = CropRecommendationApp(
        root
    )

    root.mainloop()


print("✅ GUI is ready.")
print("Run the next cell to launch the application.")

✅ GUI is ready.
Run the next cell to launch the application.


In [19]:
# =========================================================
# LAUNCH GUI
# =========================================================

launch_application()

C:\Users\surya\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\surya\anaconda3\Lib\site-p